# V7.2 Optimised Phase B Only

Run this after V7.1 Phase A. It does not reconstruct pages and does not change V7.1 Phase-A NPZ files or fingerprints.

It reads the existing V7.1 `benchmark_state.json` and reconstruction cache paths, then writes a new evaluation-cache schema. Clean ink and edge maps are computed once per page. SSIM, reconstructed ink, reconstructed edges, absolute differences, squared differences and tolerant-match maps are computed once per method and reused for all regions and connected components.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas


In [ ]:
import cv2, json, hashlib, math
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
OBSERVED_DIR=ROOT/'aligned'
OUTPUT_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'
CACHE=OUTPUT_ROOT/'persistent_cache'
METRIC_CACHE=CACHE/'evaluation'
STATE_PATH=OUTPUT_ROOT/'benchmark_state.json'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/'runs'/(RUN_NAME+'_v7_2_phase_b')
TABLES=OUT/'tables'
TABLES.mkdir(parents=True,exist_ok=True); METRIC_CACHE.mkdir(parents=True,exist_ok=True)
METHODS=['biharmonic','telea','paper_fill','paper_biharmonic_hybrid','paper_telea_hybrid']
DILATIONS=[1,2]; MIN_METRIC_COMPONENT_AREA=20; FORCE_REBUILD_EVALUATION=False
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
EVALUATION_SCHEMA='v7.2_precomputed_feature_maps'

def files(folder): return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
def find_image(folder,page):
    hits=[p for p in files(folder) if p.stem.lower()==page.lower()]
    return hits[0] if len(hits)==1 else None
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(v): return hashlib.md5(json.dumps(v,sort_keys=True).encode()).hexdigest()
def read_json(path,default):
    try: return json.loads(Path(path).read_text()) if Path(path).exists() else default
    except Exception: return default
def write_json_atomic(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,allow_nan=True,default=str)); tmp.replace(path)
def valid_npz(path,required):
    try:
        with np.load(path) as z: return set(required).issubset(z.files)
    except Exception: return False

state=read_json(STATE_PATH,None)
if not state or not state.get('items'): raise RuntimeError('V7.1 benchmark_state.json is missing or empty.')
valid_pages=sorted({v['page'] for v in state['items'].values() if v.get('reconstruction_complete')})
print('Reusable Phase-A pages:',len(valid_pages),'State:',STATE_PATH)


In [ ]:
def printed_ink(image): return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge_map(image): return cv2.Canny(image,70,170)>0
def classify_gt(clean_ink,mask):
    long_lines=cv2.morphologyEx(clean_ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long_lines.astype(np.uint8),np.ones((3,7),np.uint8))>0
    near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0; m=mask>0
    return {'all':m,'background':m&~clean_ink,'staff':m&staff,'symbol':m&clean_ink&~near,'mixed':m&clean_ink&near}
def tolerance_maps(reference,prediction,tol=1):
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    rd=cv2.dilate(reference.astype(np.uint8),k)>0; pd=cv2.dilate(prediction.astype(np.uint8),k)>0
    return {'reference':reference,'prediction':prediction,'prediction_match':prediction&rd,'reference_match':reference&pd}
def feature_bundle(clean,recon,clean_ink,clean_edges):
    signed=clean.astype(np.float32)-recon.astype(np.float32)
    _,sm=ssim(clean,recon,data_range=255,full=True)
    return {'abs_diff':np.abs(signed),'squared_diff':signed*signed,'ssim_map':sm.astype(np.float32,copy=False),
            'ink':tolerance_maps(clean_ink,printed_ink(recon)),'edge':tolerance_maps(clean_edges,edge_map(recon)),
            'dark_smudge':(recon<180)&(clean>=220),'clean_light':clean>=220}
def f1_from_maps(maps,region):
    rc=int(np.sum(maps['reference']&region))
    if rc==0:return np.nan,np.nan,np.nan
    pc=int(np.sum(maps['prediction']&region)); p=float(np.sum(maps['prediction_match']&region)/max(pc,1)); r=float(np.sum(maps['reference_match']&region)/rc)
    return p,r,float(2*p*r/max(p+r,1e-12))
def score(bundle,region):
    pixels=int(region.sum())
    if not pixels:return None
    mse=float(np.mean(bundle['squared_diff'][region])); ip,ir,if1=f1_from_maps(bundle['ink'],region); ep,er,ef1=f1_from_maps(bundle['edge'],region)
    return {'pixels':pixels,'mae':float(np.mean(bundle['abs_diff'][region])),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
            'ssim':float(np.mean(bundle['ssim_map'][region])),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1_tolerant':ef1,
            'dark_smudge_rate':float(np.sum(bundle['dark_smudge']&region)/max(np.sum(bundle['clean_light']&region),1))}


In [ ]:
metric_rows=[]; component_rows=[]
for pi,page in enumerate(valid_pages,1):
    cp=find_image(CLEAN_DIR,page); op=find_image(OBSERVED_DIR,page)
    if cp is None or op is None: print('MISSING SOURCE',page); continue
    clean=cv2.imread(str(cp),0); observed=cv2.imread(str(op),0)
    clean_ink=printed_ink(clean); clean_edges=edge_map(clean)
    for dilation in DILATIONS:
        item_id=f'{page}|d{dilation}'; item=state['items'].get(item_id,{})
        if not item.get('reconstruction_complete'): print('BLOCKED',item_id); continue
        efp=object_hash({'reconstruction':item['fingerprint'],'clean_md5':md5(cp),'evaluation_schema':EVALUATION_SCHEMA,
                         'minimum_component':MIN_METRIC_COMPONENT_AREA,'ink':[41,15],'edge':[70,170],'tolerance':1,'smudge':[180,220]})
        cache=METRIC_CACHE/f'{page}_d{dilation}_{efp}.json'; payload=read_json(cache,None) if cache.exists() and not FORCE_REBUILD_EVALUATION else None
        hit=bool(payload and payload.get('evaluation_fingerprint')==efp)
        try:
            if not hit:
                rcache=Path(item['reconstruction_cache'])
                if not valid_npz(rcache,['mask',*METHODS]): raise RuntimeError('Invalid Phase-A cache '+str(rcache))
                with np.load(rcache) as z: mask=z['mask']; outputs={m:z[m] for m in METHODS}
                regions=classify_gt(clean_ink,mask); n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
                specs=[]
                for component in range(1,n):
                    area=int(stats[component,cv2.CC_STAT_AREA])
                    if area<MIN_METRIC_COMPONENT_AREA: continue
                    x,y,w,h=map(int,stats[component,:4]); lm=labels[y:y+h,x:x+w]==component; ink_fraction=float(np.mean(clean_ink[y:y+h,x:x+w][lm]))
                    specs.append({'component':component,'category':'background' if ink_fraction<.05 else ('structure' if ink_fraction>.30 else 'mixed'),
                                  'component_area':area,'x':x,'y':y,'w':w,'h':h,'aspect_ratio':max(w,h)/max(min(w,h),1),'clean_ink_fraction':ink_fraction,'mask':lm})
                local_m=[]; local_c=[]
                for method,recon in outputs.items():
                    assert np.array_equal(recon[mask==0],observed[mask==0])
                    bundle=feature_bundle(clean,recon,clean_ink,clean_edges)
                    for rn,region in regions.items():
                        q=score(bundle,region)
                        if q: local_m.append({'page':page,'dilation':dilation,'method':method,'region':rn,**q})
                    for spec in specs:
                        x,y,w,h=spec['x'],spec['y'],spec['w'],spec['h']; local={'abs_diff':bundle['abs_diff'][y:y+h,x:x+w],'squared_diff':bundle['squared_diff'][y:y+h,x:x+w],
                            'ssim_map':bundle['ssim_map'][y:y+h,x:x+w],'dark_smudge':bundle['dark_smudge'][y:y+h,x:x+w],'clean_light':bundle['clean_light'][y:y+h,x:x+w],
                            'ink':{k:v[y:y+h,x:x+w] for k,v in bundle['ink'].items()},'edge':{k:v[y:y+h,x:x+w] for k,v in bundle['edge'].items()}}
                        q=score(local,spec['mask'])
                        if q: local_c.append({'page':page,'dilation':dilation,'method':method,**{k:v for k,v in spec.items() if k!='mask'},**q})
                payload={'version':'7.2','evaluation_schema':EVALUATION_SCHEMA,'evaluation_fingerprint':efp,'reconstruction_fingerprint':item['fingerprint'],
                         'page':page,'dilation':dilation,'metric_rows':local_m,'component_rows':local_c,'created_utc':datetime.now(timezone.utc).isoformat()}
                write_json_atomic(cache,payload)
            metric_rows.extend(payload['metric_rows']); component_rows.extend(payload['component_rows'])
            state['items'][item_id].update({'evaluation_complete':True,'evaluation_schema':EVALUATION_SCHEMA,'evaluation_fingerprint':efp,'evaluation_cache':str(cache),'updated_utc':datetime.now(timezone.utc).isoformat()})
            write_json_atomic(STATE_PATH,state); print(f'Phase B {pi}/{len(valid_pages)} {page} d{dilation}:','HIT' if hit else 'MISS')
        except Exception as e:
            state['items'][item_id].update({'evaluation_complete':False,'evaluation_schema':EVALUATION_SCHEMA,'evaluation_error':str(e)})
            state.setdefault('failures',[]).append({'phase':'B-v7.2','page':page,'dilation':dilation,'error':str(e),'utc':datetime.now(timezone.utc).isoformat()}); write_json_atomic(STATE_PATH,state); print('FAILED',item_id,e)
results=pd.DataFrame(metric_rows); components=pd.DataFrame(component_rows)
if results.empty: raise RuntimeError('No evaluation records available.')
results.to_csv(TABLES/'metrics_by_page_region.csv',index=False); components.to_csv(TABLES/'component_metrics.csv',index=False)
print('V7.2 Phase B complete:',OUT,'Metric rows:',len(results),'Component rows:',len(components))
